In [1]:
import pandas as pd # import for data reading and manipulation
import re 
from pathlib import Path

%store -r MIMIC_DIR
%store -r cohort

# OPIOID MME 
def get_mme_factor(drug_name, route, dose_unit):
    """
    Returns MME conversion factor given drug, route/formulation, and dose unit.
    Fentanyl/buprenorphine factors depend on route/formulation; other opioids use a flat factor.
    """
    drug = drug_name.lower()
    route = str(route).lower() if pd.notna(route) else ""
    unit = str(dose_unit).lower() if pd.notna(dose_unit) else ""

    # Patch (TD) factors are calibrated per mcg/hr. Some patch orders record dose_val_rx
    # as a patch *count* ("1 EA"/"PTCH") instead of the mcg/hr rate -- the true strength
    # then only lives in free-text drug name (e.g. "fentanyl 100 mcg/hr patch"), and
    # sometimes isn't recoverable at all (e.g. plain "Butrans (buprenorphine)"). Treating
    # that "1" as if it were the mcg/hr rate would silently understate dose by 10-100x.
    # Same problem for weight-based units (mcg/kg) -- can't resolve to an absolute dose
    # without patient weight, which this pipeline doesn't have. Bail out to
    # "unconvertible" instead of guessing in either case.
    is_patch_quantity_or_weight_unit = unit in ("ea", "ptch") or "kg" in unit

    if "fentanyl" in drug: # looks at the different route of administrations for fentanyl and buprenorphine
        if "patch" in drug or "td" in route or "transdermal" in route:
            if is_patch_quantity_or_weight_unit:
                return None
            return 2.4
        elif route in ("iv", "ivpb", "ivpush", "intravenous"):
            return 0.1
        else:
            return None
    if "buprenorphine" in drug:
        if "patch" in drug or "td" in route or "transdermal" in route:
            if is_patch_quantity_or_weight_unit:
                return None
            return 12.6
        elif "film" in drug or "tablet" in drug or "sl" in route or "buccal" in route:
            return 30.0
        else:
            return None

    simple_MME_FACTORS = { # other opioids
        "morphine": 1.0,
        "oxycodone": 1.5,
        "hydrocodone": 1.0,
        "hydromorphone": 4.0,
        "tramadol": 0.1,
        "codeine": 0.15,
        "methadone": 4.7,
    }
    for key, factor in simple_MME_FACTORS.items():
        if key in drug:
            return factor
    return None

OPIOID_TYPES = [ # canonical opioid name per order, for type-level comparisons
    "morphine", "fentanyl", "oxycodone", "hydrocodone", "hydromorphone",
    "tramadol", "codeine", "methadone", "buprenorphine",
]

def assign_opioid_type(drug_name):
    drug_low = drug_name.lower()
    for name in OPIOID_TYPES:
        if name in drug_low:
            return name
    return None

def compute_mme(row):
    if row["mme_factor"] is None or pd.isna(row["dose_numeric"]):
        return None
    if "patch" in row["drug"].lower(): # patch factors are mcg/hr equivalents -> convert to a daily dose
        return row["dose_numeric"] * row["mme_factor"] * 24
    return row["dose_numeric"] * row["mme_factor"]

opioids = pd.read_parquet("opioid_orders.parquet") # make the opioid prescription parquet
opioids["dose_numeric"] = pd.to_numeric(opioids["dose_val_rx"], errors="coerce")
opioids["mme_factor"] = opioids.apply(lambda r: get_mme_factor(r["drug"], r["route"], r["dose_unit_rx"]), axis=1)
opioids["opioid_type"] = opioids["drug"].apply(assign_opioid_type)

# Plausibility ceiling on patch *rates*: even a patient on several concurrent high-strength
# patches rarely exceeds a few hundred mcg/hr (this data's own non-anomalous patch rows top
# out at 400-500 mcg/hr for fentanyl). One fentanyl patch row recorded 5625 mcg/h for 3 days
# (a single-order MME of 324,000) -- almost certainly a data-entry error (e.g. a stray digit,
# or a multi-day cumulative dose entered into what should be an hourly-rate field) rather than
# a real order. Ceilings are set generously above the highest plausible values actually seen
# in this cohort's patch orders, not tuned to exclude any one specific row.
PATCH_RATE_CEILING_MCG_HR = {"fentanyl": 1000, "buprenorphine": 200}

def implausible_patch_rate(row):
    drug = row["drug"].lower()
    route = str(row["route"]).lower() if pd.notna(row["route"]) else ""
    is_patch = "patch" in drug or route in ("td", "tp") or "transdermal" in route
    if not is_patch or pd.isna(row["dose_numeric"]):
        return False
    for name, ceiling in PATCH_RATE_CEILING_MCG_HR.items():
        if name in drug:
            return row["dose_numeric"] > ceiling
    return False

opioids["implausible_patch_rate"] = opioids.apply(implausible_patch_rate, axis=1)
n_implausible = opioids["implausible_patch_rate"].sum()
print(f"Patch orders excluded as implausible rate (>ceiling): {n_implausible:,}")
opioids.loc[opioids["implausible_patch_rate"], "mme_factor"] = None

# opioid_orders.parquet was built by filtering prescriptions.csv on subject_id only
# (see cohort.ipynb), so it can include a cohort patient's orders from an admission
# that never had a qualifying cancer code. Restrict to cohort hadm_ids up front so
# every hadm_id-level aggregation below only describes cohort admissions.
opioids = opioids[opioids.hadm_id.isin(cohort.hadm_id)].copy()

n_not_converted = opioids["mme_factor"].isna().sum() # Un-usable data
print(f"Opioid orders without MME conversion: {n_not_converted:,} ({n_not_converted/len(opioids):.1%})")

# Track dose completeness *before* dropping unconvertible rows: an admission whose
# opioid orders all fail to convert (unparseable dose_val_rx, or a drug/route with no
# MME factor -- e.g. fentanyl given by a route that's neither IV nor patch) previously
# ended up with total_mme = NaN, indistinguishable from "never exposed" once that NaN
# is filled or dropped downstream. dose_fully_missing flags those admissions explicitly.
order_counts = opioids.groupby("hadm_id").size().rename("n_opioid_orders_total")
unconverted_mask = opioids["mme_factor"].isna() | opioids["dose_numeric"].isna()
unconverted_counts = opioids[unconverted_mask].groupby("hadm_id").size().rename("n_opioid_orders_unconverted")

dose_completeness = (
    pd.concat([order_counts, unconverted_counts], axis=1)
    .fillna({"n_opioid_orders_unconverted": 0})
    .reset_index()
)
dose_completeness["n_opioid_orders_unconverted"] = dose_completeness["n_opioid_orders_unconverted"].astype(int)
dose_completeness["dose_fully_missing"] = (
    dose_completeness["n_opioid_orders_total"] == dose_completeness["n_opioid_orders_unconverted"]
)
n_dose_fully_missing = dose_completeness["dose_fully_missing"].sum()
print(f"Admissions with opioid orders but 0 convertible to an MME dose: {n_dose_fully_missing:,}")
%store dose_completeness

opioids_clean = opioids.dropna(subset=["dose_numeric", "mme_factor"]).copy()
opioids_clean["mme_dose"] = opioids_clean.apply(compute_mme, axis=1)

print(f"{len(opioids_clean):,}")

print("Orders per opioid type:")
print(opioids_clean["opioid_type"].value_counts())

mme_summary = opioids_clean.groupby("hadm_id").agg( # opioid information
    total_mme = ("mme_dose", "sum"),
    n_opioid_orders = ("mme_dose", "count"),
    max_single_dose_mme = ("mme_dose", "max"),
).reset_index()

%store mme_summary
%store opioids_clean

Patch orders excluded as implausible rate (>ceiling): 1
Opioid orders without MME conversion: 5,407 (2.2%)
Admissions with opioid orders but 0 convertible to an MME dose: 9,287
Stored 'dose_completeness' (DataFrame)
153,259
Orders per opioid type:
opioid_type
oxycodone        52940
hydromorphone    51458
morphine         28875
tramadol          7466
fentanyl          6992
methadone         3923
codeine            785
hydrocodone        624
buprenorphine      196
Name: count, dtype: int64
Stored 'mme_summary' (DataFrame)
Stored 'opioids_clean' (DataFrame)


In [2]:
opioids_clean["duration_days"] = ( # look for opioid mme conversion and convert
    (opioids_clean["stoptime"] - opioids_clean["starttime"]).dt.total_seconds()/86400
    ).clip(lower=1/24)

opioids_clean["daily_mme"] = opioids_clean["mme_dose"] / opioids_clean["duration_days"]

daily_mme_summary = opioids_clean.groupby("hadm_id").agg( # look for mean and max mme
    mean_daily_mme = ("daily_mme", "mean"),
    max_daily_mme = ("daily_mme", "max"),
).reset_index()

%store daily_mme_summary

Stored 'daily_mme_summary' (DataFrame)


In [3]:
patch_rows = opioids_clean[opioids_clean["drug"].str.contains("patch", case=False, na=False)] # check to make sure computing correctly
non_patch_rows = opioids_clean[~opioids_clean["drug"].str.contains("patch", case=False, na=False)]

fentanyl_rows = opioids_clean[opioids_clean.drug.str.contains("fentanyl", case=False, na=False)]
print(fentanyl_rows[["drug", "route", "dose_val_rx", "dose_unit_rx"]].drop_duplicates())

bup_rows = opioids_clean[opioids_clean.drug.str.contains("bupren", case=False, na=False)]
print(bup_rows[["drug", "route", "dose_val_rx", "dose_unit_rx"]].drop_duplicates())


                    drug route dose_val_rx dose_unit_rx
400       Fentanyl Patch    TD          25        mcg/h
405       Fentanyl Patch    TD          12        mcg/h
407       Fentanyl Patch    TD          50        mcg/h
763     Fentanyl Citrate    IV          25          mcg
1245    Fentanyl Citrate    IV          75          mcg
...                  ...   ...         ...          ...
214471          fentaNYL    TD          75       mcg/hr
229166          fentaNYL    TD          50       mcg/hr
239641  Fentanyl Citrate    IV       162.5          mcg
246222  Fentanyl Citrate    IV         500          mcg
273966    Fentanyl Patch    TP           1       mcg/hr

[79 rows x 4 columns]
                                             drug        route dose_val_rx  \
14218       Buprenorphine-Naloxone Film (8mg-2mg)           SL           1   
14219     Buprenorphine-Naloxone Tablet (8mg-2mg)           SL           1   
16302            Buprenorphine-Naloxone (8mg-2mg)           SL         

In [4]:
# check the most extreme single-order MME values before they feed a model --
# these could be legitimate (e.g. methadone maintenance, high-tolerance patients)
# or a rate-vs-total/unit entry error; worth a manual look either way
outlier_cols = ["hadm_id", "drug", "route", "dose_val_rx", "dose_unit_rx", "starttime", "stoptime", "mme_dose"]
print(opioids_clean.sort_values("mme_dose", ascending=False)[outlier_cols].head(20).to_string(index=False))

 hadm_id               drug route dose_val_rx dose_unit_rx           starttime            stoptime  mme_dose
24934171 HYDROmorphone P.F.    ED        7500          mcg 2178-03-23 15:00:00 2178-03-23 22:00:00   30000.0
24934171 HYDROmorphone P.F.    ED        7500          mcg 2178-03-22 14:00:00 2178-03-23 14:00:00   30000.0
25687562 HYDROmorphone P.F.    ED        7500          mcg 2114-03-06 17:00:00 2114-03-06 16:00:00   30000.0
25687562 HYDROmorphone P.F.    ED        7500          mcg 2114-03-06 19:00:00 2114-03-08 10:00:00   30000.0
25687562 HYDROmorphone P.F.    ED        7500          mcg 2114-03-06 16:00:00 2114-03-06 16:00:00   30000.0
25687562 HYDROmorphone P.F.    ED        7500          mcg 2114-03-06 17:00:00 2114-03-06 16:00:00   30000.0
25687562 HYDROmorphone P.F.    ED        7500          mcg 2114-03-06 17:00:00 2114-03-06 18:00:00   30000.0
25478987 HYDROmorphone P.F.    ED        7500          mcg 2127-09-13 13:00:00 2127-09-14 13:00:00   30000.0
24469601     Fentan

In [5]:
# DRUG INTERACTIONS (benzodiazepines, gabapentinoids, antipsychotics, muscle relaxants, antihistamines, z-drugs)
MIMIC_DIR = Path("/home/emma/data/physionet.org/files/mimiciv/3.1/hosp/")

# NOTE: the adjacent string literals below must each end in "|" -- a missing "|" here
# silently merges two drug names into one unmatchable token (e.g. "pregabalinzolpidem"),
# which is what happened previously and meant pregabalin/zolpidem were never detected.
sedative_pattern = re.compile(
    r"(lorazepam|midazolam|diazepam|alprazolam|clonazepam|gabapentin|pregabalin|"
    r"zolpidem|quetiapine|olanzapine|cyclobenzaprine|diphenhydramine|hydroxyzine)",
    re.IGNORECASE
) # specific sedatives to filter from the cohort

cohort_with_opioids = pd.read_parquet("cohort_with_opioids.parquet") 
cohort_opioid_subject_ids = set(cohort_with_opioids.subject_id.unique())
chunks = []
chunk_size = 1_000_000 

reader = pd.read_csv(MIMIC_DIR / "prescriptions.csv", chunksize=chunk_size, # same thing as with opioid reader
                     dtype = {"subject_id": "int64", "hadm_id": "Int64"},
                     parse_dates=["starttime", "stoptime"],
                     usecols = ["subject_id", "hadm_id", "drug", "dose_val_rx", 
                                "dose_unit_rx", "route", "starttime", "stoptime"],
                    low_memory=False,
)

for i, chunk in enumerate(reader): # finds sedative use in opioid cohort
    chunk = chunk[chunk.subject_id.isin(cohort_opioid_subject_ids)]
    chunk = chunk[chunk.drug.str.contains(sedative_pattern, case=False, na=False, regex=True)]
    if len(chunk):
        chunks.append(chunk)
    if i % 10 == 0:
        print(f"Processed {i} chunks, sedative rows: {sum(len(c) for c in chunks):,}")

sedatives = pd.concat(chunks, ignore_index=True)
print(f"Total sedative rows: {len(sedatives):,}")

sedatives.to_parquet("sedative_orders.parquet")
%store sedatives

cohort_with_sedatives = cohort_with_opioids.merge(
    sedatives, on = ["subject_id", "hadm_id"], how = "left"
)
cohort_with_sedatives.to_parquet("cohort_with_sedatives.parquet") # saves parquet file
print("Done. Saved cancer_cohort.parquet, sedative_orders.parquet, and cohort_with_sedatives.parquet")

sedative_flag = (
    sedatives.groupby("hadm_id")
    .size()
    .reset_index(name="n_sedative_orders")
)
sedative_flag["concurrent_sedative_use"] = sedative_flag["n_sedative_orders"] > 0

%store sedative_flag

/home/emma/Defining Cohort/ipykernel_1523158/717749319.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.contains(sedative_pattern, case=False, na=False, regex=True)]


Processed 0 chunks, sedative rows: 10,118


/home/emma/Defining Cohort/ipykernel_1523158/717749319.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.contains(sedative_pattern, case=False, na=False, regex=True)]
/home/emma/Defining Cohort/ipykernel_1523158/717749319.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.contains(sedative_pattern, case=False, na=False, regex=True)]
/home/emma/Defining Cohort/ipykernel_1523158/717749319.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.contains(sedative_pattern, case=False, na=False, regex=True)]
/home/emma/Defining Cohort/ipykernel_1523158/717749319.py:28: UserWarning: This pattern is interpreted as a regular expression, and has 

Processed 10 chunks, sedative rows: 115,441


/home/emma/Defining Cohort/ipykernel_1523158/717749319.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.contains(sedative_pattern, case=False, na=False, regex=True)]
/home/emma/Defining Cohort/ipykernel_1523158/717749319.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.contains(sedative_pattern, case=False, na=False, regex=True)]
/home/emma/Defining Cohort/ipykernel_1523158/717749319.py:28: UserWarning: This pattern is interpreted as a regular expression, and has match groups. To actually get the groups, use str.extract.
  chunk = chunk[chunk.drug.str.contains(sedative_pattern, case=False, na=False, regex=True)]
/home/emma/Defining Cohort/ipykernel_1523158/717749319.py:28: UserWarning: This pattern is interpreted as a regular expression, and has 

Processed 20 chunks, sedative rows: 215,179
Total sedative rows: 215,179
Stored 'sedatives' (DataFrame)
Done. Saved cancer_cohort.parquet, sedative_orders.parquet, and cohort_with_sedatives.parquet
Stored 'sedative_flag' (DataFrame)


In [6]:
def has_temporal_overlap(opioid_intervals, sedative_intervals):
    for o_start, o_end in opioid_intervals:
        for s_start, s_end in sedative_intervals:
            if o_start <= s_end and s_start <= o_end:
                return True
    return False    

opioid_intervals = opioids_clean.groupby("hadm_id").apply(
    lambda d: list(zip(d["starttime"], d["stoptime"]))
)

sedative_intervals = sedatives.groupby("hadm_id").apply(
    lambda d: list(zip(d["starttime"], d["stoptime"]))
)

overlap_flag = pd.DataFrame({
    "hadm_id": opioid_intervals.index
}).merge(opioid_intervals.rename("o_int"), on = "hadm_id") \
.merge(sedative_intervals.rename("s_int"), on = "hadm_id", how = "left")

overlap_flag["true_concurrent_use"] = overlap_flag.apply(
    lambda t: has_temporal_overlap(t["o_int"], t["s_int"]) if isinstance(t["s_int"], list) else False,
    axis = 1
)
%store overlap_flag

Stored 'overlap_flag' (DataFrame)


In [7]:
%store -r opioids_clean

# OPIOID TOLERANCE: is this order a patient's *first ever recorded* opioid order,
# or have they already had one earlier (any admission)? Computed per-order so it
# can be attached to each dose downstream without leaking future information.
opioids_clean = opioids_clean.sort_values(["subject_id", "starttime"]).copy()
opioids_clean["prior_opioid_orders"] = opioids_clean.groupby("subject_id").cumcount()
opioids_clean["opioid_naive"] = opioids_clean["prior_opioid_orders"] == 0

print(f"Orders that are a patient's first recorded opioid exposure: {opioids_clean['opioid_naive'].mean():.1%}")

%store opioids_clean

Orders that are a patient's first recorded opioid exposure: 14.4%
Stored 'opioids_clean' (DataFrame)
